# 01 · Dataset, vocabulary & eval-path check
Builds manifests + **train-only** gloss vocab, the debug **smoke** split,
verifies official split counts, and inspects CorrNet's evaluator so you can
bind it. **Gate:** counts = 5672/540/629, vocab ≈ 1081 (+blank).

In [ ]:
# --- Colab bootstrap (run first in every notebook) ---
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT = '/content/drive/MyDrive/cslr_phoenix/project'   # <- where this code lives
sys.path.append(PROJECT)
%cd $PROJECT

!pip -q install pyyaml
from src.utils import load_config
cfg = load_config('config.yaml')
print('config loaded:', cfg['project']['name'])


## Parse corpus + verify official counts

In [ ]:
from src.vocab import parse_corpus
from src.utils import save_json, p

items = {s: parse_corpus(cfg, s) for s in cfg['dataset']['splits']}
counts = {s: len(v) for s, v in items.items()}
print('counts:', counts)
exp = cfg['dataset']['expected_counts']
for s in exp:
    assert counts[s] == exp[s], f"{s}: got {counts[s]}, expected {exp[s]} — wrong split/release?"
print('OK — full official PHOENIX-2014 split confirmed.')

## Build train-only vocabulary (no leakage)

In [ ]:
from src.vocab import build_vocab, save_vocab

gloss2id = build_vocab(items['train'], blank_index=cfg['vocab']['blank_index'])
save_vocab(gloss2id, p(cfg, 'manifests') / 'vocab.json')
print('vocab size (excl. blank):', len(gloss2id))

# dev/test glosses unseen in train -> will be unavoidable recognition errors; just report.
train_set = set(gloss2id)
for s in ('dev', 'test'):
    oov = {g for it in items[s] for g in it['glosses']} - train_set
    print(f'{s}: {len(oov)} OOV gloss types (expected small)')

## Save manifests + smoke split

In [ ]:
import random
for s in cfg['dataset']['splits']:
    save_json(items[s], p(cfg, 'manifests') / f'{s}.json')

random.seed(cfg['project']['seed'])
tr_ids = [it['id'] for it in items['train']]
dv_ids = [it['id'] for it in items['dev']]
random.shuffle(tr_ids); random.shuffle(dv_ids)
save_json(tr_ids[:cfg['dataset']['smoke_train']], p(cfg, 'manifests') / 'smoke_train_ids.json')
save_json(dv_ids[:cfg['dataset']['smoke_dev']],   p(cfg, 'manifests') / 'smoke_dev_ids.json')
print('manifests + smoke split saved to', p(cfg, 'manifests'))

## Gloss-length stats (feeds the T ≥ 2·L check in NB 04)

In [ ]:
import numpy as np
for s in cfg['dataset']['splits']:
    L = np.array([len(it['glosses']) for it in items[s]])
    print(f'{s}: glosses/sentence  min {L.min()}  mean {L.mean():.1f}  max {L.max()}')

## Inspect the official evaluator (binding prep)
Clone CorrNet into the VM first if needed:
```
!git clone https://github.com/hulianyuyy/CorrNet /content/CorrNet
```

In [ ]:
from src.official_eval_adapter import inspect_official_eval
inspect_official_eval(cfg['paths']['corrnet_repo'])
# Use the printed function signatures to bind official_eval() in
# src/official_eval_adapter.py, then certify against the CorrNet checkpoint.